In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from time import sleep, time

project_id = 37504063
BT_LOW = '65db6297918a392778ffa30171019af9'   # overnight ETF rotation, LIQ top-half + LOW-vol tilt (current main.py)
BT_HIGH = '46c07f7c29bdbe9e81f2c166b64298af'  # same strategy, HIGH-vol tilt variant

def read_chart(backtest_id, chart_name, count=10000):
    for attempt in range(10):
        result = api.read_backtest_chart(project_id, chart_name, 0, int(time()), count, backtest_id)
        if result.success:
            return result.chart
        sleep(10)
    raise RuntimeError(f'Failed to read backtest {chart_name} chart after 10 attempts')

def to_series(chart, series_name, selector=lambda x: x.y):
    return pd.Series({v.time: selector(v) for v in chart.series[series_name].values})

def load_curve(backtest_id):
    equity = to_series(read_chart(backtest_id, 'Strategy Equity'), 'Equity', selector=lambda x: x.close)
    benchmark = to_series(read_chart(backtest_id, 'Benchmark'), 'Benchmark')
    drawdown = to_series(read_chart(backtest_id, 'Drawdown'), 'Equity Drawdown')
    df = pd.DataFrame({'Equity': equity, 'Benchmark': benchmark, 'Drawdown': drawdown}).ffill()
    idx = pd.to_datetime(df.index)
    if idx.tz is None:
        idx = idx.tz_localize('UTC')
    df.index = idx.tz_convert('US/Eastern').tz_localize(None)
    df = df[~df.index.duplicated(keep='last')].sort_index()
    df['Return'] = df['Equity'].pct_change().fillna(0.0)
    return df

low = load_curve(BT_LOW)
high = load_curve(BT_HIGH)
print(f'low-vol samples: {len(low)} | high-vol samples: {len(high)}')
print(low[['Equity', 'Drawdown']].head(3))
print(low[['Equity', 'Drawdown']].tail(3))



NameError: name 'api' is not defined

In [ ]:
def curve_stats(df: 'pd.DataFrame', label: str) -> None:
    r = df['Return']
    years = (df.index[-1] - df.index[0]).days / 365.25
    total = df['Equity'].iloc[-1] / df['Equity'].iloc[0] - 1
    cagr = (1 + total) ** (1 / years) - 1
    ann_vol = r.std() * np.sqrt(252)
    sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else float('nan')
    max_dd = (df['Equity'] / df['Equity'].cummax() - 1).min()
    win = (r > 0).mean()
    print(f'{label}: total {total:.2%} | CAGR {cagr:.2%} | vol {ann_vol:.2%} | Sharpe {sharpe:.2f} | maxDD {max_dd:.2%} | win {win:.2%} | samples {len(r)}')

print('From results-page chart data (equity net of fees):')
curve_stats(low, 'LOW-VOL tilt ')
curve_stats(high, 'HIGH-VOL tilt')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(low.index, low['Equity'] / low['Equity'].iloc[0], label='Low-vol tilt (net of fees)', lw=1.2)
ax.plot(high.index, high['Equity'] / high['Equity'].iloc[0], label='High-vol tilt (net of fees)', lw=1.2)
ax.plot(low.index, low['Benchmark'] / low['Benchmark'].iloc[0], color='grey', alpha=0.6, label='SPY buy-and-hold (benchmark, no fees)', lw=1.0)
ax.set_yscale('log')
ax.set_title('Overnight ETF rotation: growth of $1 (2010 - 2026, liquidity top-half filter)')
ax.set_ylabel('Multiple of initial capital (log scale)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

plt.show()

In [ ]:
# Drawdown computed from the equity curve (matches the results-page stats).
low_dd = (low['Equity'] / low['Equity'].cummax() - 1) * 100
high_dd = (high['Equity'] / high['Equity'].cummax() - 1) * 100
# S&P 500 buy-and-hold drawdown from the backtest benchmark series (SPY price, no dividends/fees).
spy_dd = (low['Benchmark'] / low['Benchmark'].cummax() - 1) * 100
spy_max_dd = spy_dd.min()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(low_dd.index, low_dd, 0, alpha=0.5, label='Low-vol tilt')
ax.fill_between(high_dd.index, high_dd, 0, alpha=0.4, color='firebrick', label='High-vol tilt')
ax.plot(spy_dd.index, spy_dd, color='grey', lw=1.0, alpha=0.8, label='S&P 500 buy-and-hold (SPY price, no fees)')
ax.axhline(spy_max_dd, color='grey', lw=1.0, ls='--', label=f'S&P 500 max drawdown ({spy_max_dd:.1f}%)')
ax.set_title('Equity drawdown from peak (net of fees)')
ax.set_ylabel('% below peak')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print(f'S&P 500 buy-and-hold max drawdown: {spy_max_dd:.2f}%')

In [ ]:
def rolling_sharpe(r: 'pd.Series', window: int) -> 'pd.Series':
    mu = r.rolling(window).mean()
    sigma = r.rolling(window).std()
    return mu / sigma * np.sqrt(252)

# ~1 trading year of samples; shrinks only if the API downsampled the chart.
window = 252 if len(low) > 3000 else max(30, len(low) // 12)
# SPY returns from the backtest benchmark series (SPY close, price only, no dividends/fees)
spy_returns = low['Benchmark'].pct_change().fillna(0.0)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(low.index, rolling_sharpe(low['Return'], window), label='Low-vol tilt', lw=1.2)
ax.plot(high.index, rolling_sharpe(high['Return'], window), label='High-vol tilt', lw=1.2, alpha=0.85)
ax.plot(low.index, rolling_sharpe(spy_returns, window), label='S&P 500 (SPY price benchmark, no fees)', lw=1.2, color='grey')
ax.axhline(0, color='black', lw=0.8)
ax.set_title(f'Rolling annualized Sharpe ratio ({window}-sample window, net of fees)')
ax.set_ylabel('Sharpe ratio')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print(f'rolling window: {window} samples (~1 trading year when sampled daily)')

In [ ]:
vol_window = 63 if len(low) > 3000 else max(20, len(low) // 24)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(low.index, low['Return'].rolling(vol_window).std() * np.sqrt(252) * 100, label='Low-vol tilt', lw=1.2)
axes[0].plot(high.index, high['Return'].rolling(vol_window).std() * np.sqrt(252) * 100, label='High-vol tilt', lw=1.2, alpha=0.85)
axes[0].set_title(f'Rolling annualized volatility ({vol_window}-sample window)')
axes[0].set_ylabel('Annualized vol (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)

lo = min(low['Return'].min(), high['Return'].min()) * 100
hi = max(low['Return'].max(), high['Return'].max()) * 100
bins = np.linspace(lo, hi, 60)
axes[1].hist(low['Return'] * 100, bins=bins, alpha=0.6, density=True, label='Low-vol tilt')
axes[1].hist(high['Return'] * 100, bins=bins, alpha=0.5, density=True, label='High-vol tilt')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Daily equity return distribution')
axes[1].set_xlabel('Daily return (%)')
axes[1].set_ylabel('Density')
axes[1].legend()
plt.show()

In [ ]:
# Monthly net returns of the low-vol tilt (the current main.py variant), %
r = low['Return']
monthly = (1 + r).groupby([r.index.year, r.index.month]).prod() - 1
matrix = (monthly.unstack() * 100).reindex(columns=range(1, 13))

fig, ax = plt.subplots(figsize=(13, 8))
im = ax.imshow(matrix.values, cmap='RdYlGn', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set_title('Low-vol tilt: monthly net returns (%)')
fig.colorbar(im, ax=ax, shrink=0.7, label='%')
plt.show()

annual = ((1 + r).groupby(r.index.year).prod() - 1) * 100
print('Annual net returns (%) - low-vol tilt:')
print(annual.round(2).to_string())